# Run review

Compare saved training runs and evaluation reports, export figures, and find recorded races for review.
This is P3's analysis increment; synchronized playback controls and annotation editing are still pending.

**Setup:** select the repository's `venv/bin/python` kernel in your IDE. For a fresh environment, run
`venv/bin/python -m pip install -e ".[analysis]"` from the repository root.
Launch in a browser with `venv/bin/python -m jupyter lab notebooks/run_review.ipynb`.
Run the cells in order, then edit the selection cell to choose runs. No training or model loading occurs.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "run.py").exists() and (p / "src" / "analysis").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the F110_MARL repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display
from src.analysis.run_review import (
    discover_runs, load_run, combine, latest_evaluations, summarize_races,
    summarize_agents, aggregate_training_seeds, filter_clips, replay_command,
)
from src.analysis.plots import (
    learning_curves, optimizer_curves, evaluation_curves, outcome_plots,
    finish_time_plots, seed_variation_plots, export_figures,
)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
OUTPUT_ROOT = ROOT / "outputs"


## Available runs
Discovery reads small metadata/log files, never checkpoints or trajectory chunks.
Folders are distinct identities even when their saved `run_id` is the same.
Older runs without P0 race records remain visible, but cannot supply the new race-outcome plots.


In [ ]:
run_paths = discover_runs(OUTPUT_ROOT)
# Load only selected runs below. The inventory here reads metadata and small availability checks.
from src.analysis.run_review import read_json
inventory = []
for path in run_paths:
    snap = read_json(path / "config_snapshot.json")
    report = read_json(path / "evaluation_report.json")
    provenance = snap.get("provenance") or report.get("checkpoint_provenance") or {}
    inventory.append({
        "run": str(path.relative_to(OUTPUT_ROOT)), "scenario": provenance.get("scenario_name"),
        "training_seed": provenance.get("seed"),
        "training_races": (path / "race_metrics.jsonl").exists(),
        "evaluation": (path / "evaluation_report.json").exists() or (path / "evaluation_history.jsonl").exists(),
        "recorded_clips": (path / "behavior" / "clips.jsonl").exists(),
    })
display(pd.DataFrame(inventory))


## Select runs
Set `RUN_NAMES` to folder names from the inventory. With no selection, a single recent P0 run is opened
as a preview; select matched runs explicitly for comparisons. Nested output folders are supported.
Use `EXTRA_DATASETS` for recordings written with a custom `--dataset-dir`.

The existing smoke runs test logging and recording; their short horizons do not establish policy quality.


In [ ]:
RUN_NAMES = []  # e.g. ["p2_recording_verified_enabled", "metrics_monitoring_smoke_eval_verified"]
EXTRA_DATASETS = {}  # e.g. {"my_run": [ROOT / "datasets" / "my_recording"]}
SMOOTHING_POINTS = 1  # Reporting barriers, weighted by completed-race count; 1 = no smoothing.
EXPORT = False
EXPORT_DIR = OUTPUT_ROOT / "analysis" / "run_review"

if RUN_NAMES:
    selected_paths = [OUTPUT_ROOT / name for name in RUN_NAMES]
    missing = [str(p) for p in selected_paths if p.resolve() not in run_paths]
    if missing:
        raise ValueError(f"No saved run artifacts found: {missing}")
else:
    candidates = [p for p in run_paths if (p / "race_metrics.jsonl").exists()]
    selected_paths = sorted(candidates, key=lambda p: (p / "race_metrics.jsonl").stat().st_mtime)[-1:]
    if not selected_paths:
        selected_paths = run_paths[-1:]
    print("Preview selection:", [str(p.relative_to(OUTPUT_ROOT)) for p in selected_paths])

runs = [load_run(p, label=str(p.relative_to(OUTPUT_ROOT)),
                 dataset_dirs=EXTRA_DATASETS.get(str(p.relative_to(OUTPUT_ROOT)), ())) for p in selected_paths]
metadata = pd.DataFrame([r.metadata for r in runs])
races, agents, updates, evaluations, clips = [combine(runs, table) for table in
                                            ("races", "agents", "updates", "evaluations", "clips")]
display(metadata)
display(evaluations)
if races.empty:
    print("No P0 race records in this selection. Update diagnostics may still be available.")
figures = {}


## Training and checkpoint-selection curves
Training plots use completed episodes and the aggregate environment-step count at the reporting barrier,
not an invented exact global completion time. Continuous and finite training get separate figures.
Checkpoint-selection evaluation gets separate curves, grouped by recorded evaluation conditions.
Missing losses or diagnostics are displayed as unavailable.


In [ ]:
figures.update(learning_curves(races, window=SMOOTHING_POINTS))
figures.update(optimizer_curves(updates))
figures.update(evaluation_curves(races))
plt.show()


## Evaluation outcomes and finish times
The comparison uses the **latest checkpoint-selection evaluation** and any selected **standalone report**.
Selection, final, and custom protocols remain separate. Tables retain checkpoint identity, counts,
and a fingerprint of recorded seeds, map assignments, horizon, and physics conditions.
Compare provenance too: matching fingerprints do not prove every opponent or unrecorded setting matches.

Finish times include only clean learner finishers; plots show completion counts and sample sizes.
The per-car table includes learners and opponents. Training summaries describe all completed logged
training episodes, rather than just recorded clips, and are kept in a separate table.


In [ ]:
training = races.loc[races.phase == "training"] if not races.empty else races
chosen_eval = latest_evaluations(races)
training_summary = summarize_races(training)
evaluation_summary = summarize_races(chosen_eval)
overall_evaluation = summarize_races(chosen_eval, by_map=False)
if not chosen_eval.empty:
    selected_ids = chosen_eval[["run_key", "evaluation_id"]].drop_duplicates()
    eval_agents = agents.merge(selected_ids, on=["run_key", "evaluation_id"], validate="many_to_one")
else:
    eval_agents = pd.DataFrame()
per_car = summarize_agents(eval_agents)
display(evaluation_summary)
display(overall_evaluation)
display(per_car)
display(training_summary)
figures.update(outcome_plots(evaluation_summary))
figures.update(finish_time_plots(eval_agents))
plt.show()


## Variation across training seeds
Assign each selected evaluation run to an experiment arm below. Choose one run/checkpoint per training
seed in each arm and protocol. Repeated runs of the same training seed are not independent seeds.
Different recorded evaluation conditions stay in separate groups.

Each seed contributes one run-level value, regardless of evaluation episode count. Error bars are
**sample standard deviation across training seeds**, not episode-level uncertainty or confidence
intervals. With one measured seed, SD is missing. Inspect the outcome tables for race denominators.


In [ ]:
COMPARISON_GROUPS = {}  # Folder label -> arm, e.g. {"base_seed1/eval": "base", "base_seed2/eval": "base"}
unknown = set(COMPARISON_GROUPS) - {r.metadata["run"] for r in runs}
if unknown:
    raise ValueError(f"Comparison groups refer to unselected runs: {sorted(unknown)}")
groups = {r.metadata["run_key"]: COMPARISON_GROUPS[r.metadata["run"]]
          for r in runs if r.metadata["run"] in COMPARISON_GROUPS}
seed_summary = aggregate_training_seeds(evaluation_summary, groups)
display(seed_summary)
figures.update(seed_variation_plots(seed_summary))
plt.show()


## Find recorded races and event clips
This index reads clip metadata only. Representative samples and event-selected clips remain visibly
separate: event clips cannot estimate behavior frequencies. Partial/open clips remain identifiable.
Outcome filters use completed training race facts when available; missing outcomes do not count as failures.

Choose a row and copy the generated replay command into a terminal with a graphical display.
Review uses the recording's policy-version interval; a corresponding saved checkpoint may not exist.
In-notebook synchronized playback, seeking, and annotation editing are the next P3 increment.


In [ ]:
CLIP_FILTERS = dict(run=None, map_id=None, kind=None, event=None, agent=None, outcome=None)
# kind: "representative_race" / "event_clip"; event: "candidate_pass", "terminal", etc.
# outcome: "both_finished", "first_place", "sweep", "any_learner_collision_dnf".
filtered_clips = filter_clips(clips, **CLIP_FILTERS).reset_index(drop=True)
columns = [c for c in ["run", "clip_id", "map_id", "kind", "retention_reasons", "complete",
    "status", "end_reason", "policy_version_start", "policy_version_end", "episode_id",
    "start_physics_index", "end_physics_index", "both_finished", "any_learner_collision_dnf"]
    if c in filtered_clips]
display(filtered_clips[columns])
CLIP_ROW = 0
if not filtered_clips.empty:
    print(replay_command(filtered_clips.iloc[CLIP_ROW], ROOT, speed=1))
else:
    print("No matching clips. Record with --record-races, or set EXTRA_DATASETS for a custom location.")


## Export analysis
Set `EXPORT = True` in the selection cell to write CSV tables and standalone PNG/PDF figures.
The manifest records selected folders, configuration/checkpoint hashes, evaluation conditions, smoothing,
and seed-group assignments. Exports use the files as read during this notebook execution;
rerun after training writes new data. Raw logs and recordings remain the source of truth.


In [ ]:
if EXPORT:
    import json
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tables = {"runs": metadata, "evaluations": evaluations, "training_summary": training_summary,
              "evaluation_by_map": evaluation_summary, "evaluation_overall": overall_evaluation,
              "per_car": per_car, "training_seed_summary": seed_summary, "clips": filtered_clips}
    for name, table in tables.items():
        table.to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    export_figures(figures, EXPORT_DIR / "figures")
    manifest = {"runs": [r.metadata for r in runs], "smoothing_reporting_points": SMOOTHING_POINTS,
                "comparison_groups": COMPARISON_GROUPS, "clip_filters": CLIP_FILTERS,
                "evaluation_snapshots": json.loads(evaluations.to_json(orient="records"))}
    (EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
    print(f"Exported {len(figures)} figures and {len(tables)} tables to {EXPORT_DIR}")
